# 04 — View saved sequential reconstructions

Select the scan pair and a run from 03. Only metadata are read to build the catalog; selecting a frame loads one saved ROI or one mode. No raw detector files or retrieval are needed. Differences use their saved focus, shifts and ROI; reconstructions use the saved PNG focus. Color limits come from the run exports, so browsing does not autoscale each frame.


In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

HERE = Path.cwd()
if not (HERE / "reconstruction_inputs.py").is_file():
    HERE = HERE / "aperiodic_ajajas"
sys.path.insert(0, str(HERE.parent))
from aperiodic_ajajas.result_io import result_catalog, load_result_image

SATURATED_SCAN = 2923
LOOP_SCAN = 2924
RUN_DIR = None  # Set to a specific timestamp directory to inspect an older run.
ROOT = HERE / "processed" / f"scans_{SATURATED_SCAN}_{LOOP_SCAN}" / "sequential"
if RUN_DIR is None:
    runs = sorted(path for path in ROOT.glob("*") if path.is_dir() and any(path.glob("*/result.h5")))
    if not runs:
        raise FileNotFoundError(f"No sequential results in {ROOT}")
    RUN_DIR = runs[-1]
RUN_DIR = Path(RUN_DIR)
records = result_catalog(RUN_DIR)
if not records:
    raise ValueError(f"No result.h5 files in {RUN_DIR}")
print("Run:", RUN_DIR, "—", len(records), "saved states")
for row in records:
    print(f"{row['label']:15s} scan={row['source_scan']} current={row['current_A']:.6g} A, modes={row['modes']}")


## Browse a saved state
Use the controls below. `difference` shows the primary-mode state minus saturation; `reconstruction` shows the selected individual mode, including saturation. Unknown currents in older files display as `nan`, not an assumed value. If a run was interrupted before image export, rerun 03's export helpers to create its fixed color limits.


In [ ]:
def plot_saved(path, stage="warmup", kind="difference", mode=0):
    try:
        image, metadata, settings = load_result_image(path, stage, kind, mode)
    except (ValueError, IndexError) as error:
        print(error)
        return
    limits = settings.get("color_limits")
    if limits is None:
        print("This run has no fixed color limits yet. Finish/re-run 03's PNG export first.")
        return
    fig, axes = plt.subplots(2, 2, figsize=(9, 8), layout="constrained")
    for ax, (name, component, cmap) in zip(axes.flat, [
        ("amplitude", np.abs, "gray"), ("phase", np.angle, "twilight"),
        ("real", np.real, "gray"), ("imaginary", np.imag, "gray")]):
        low, high = limits[name]
        artist = ax.imshow(component(image), cmap=cmap, vmin=low, vmax=high,
                           interpolation="nearest", origin="upper")
        ax.set_title(name)
        fig.colorbar(artist, ax=ax)
    current = float(metadata.get("current_A", np.nan))
    fig.suptitle(f"{Path(path).parent.name} | {current:.6g} A | {stage} {kind} | mode {mode}")
    plt.show()
    print("Source file:", metadata.get("source_file", "unknown"))
    print("Current source:", metadata.get("current_source", "legacy metadata"))
    print("Array:", image.shape, "| source scan:", metadata.get("source_scan", "legacy metadata"))

import ipywidgets as widgets
frame = widgets.Dropdown(options=[(f"{r['label']} ({r['current_A']:.5g} A)", str(r['path'])) for r in records],
                         description="State:", layout=widgets.Layout(width="550px"))
stage = widgets.Dropdown(options=["warmup", "poisson"], description="Stage:")
kind = widgets.Dropdown(options=["reconstruction", "difference"], description="Image:")
mode = widgets.IntSlider(min=0, max=max(row['modes'] for row in records)-1, value=0,
                         description="Mode:", continuous_update=False)
controls = widgets.VBox([frame, widgets.HBox([stage, kind, mode])])
view = widgets.interactive_output(plot_saved, dict(path=frame, stage=stage, kind=kind, mode=mode))
display(controls, view)


## Optional native-pixel PNG export of the selected image
Set `EXPORT_SELECTED = True` and run this cell. It uses the stored fixed limits and writes only array pixels, without figure borders or interpolation. Full-precision complex data remain in HDF5.


In [ ]:
EXPORT_SELECTED = False
if EXPORT_SELECTED:
    image, metadata, settings = load_result_image(frame.value, stage.value, kind.value, mode.value)
    if "color_limits" not in settings:
        raise ValueError("Run 03's export pass to establish fixed limits first")
    destination = RUN_DIR / "viewer_exports"
    destination.mkdir(exist_ok=True)
    for name, component, cmap in [("amplitude", np.abs, "gray"), ("phase", np.angle, "twilight"),
                                  ("real", np.real, "gray"), ("imaginary", np.imag, "gray")]:
        path = destination / f"{Path(frame.value).parent.name}_{stage.value}_{kind.value}_mode_{mode.value:02d}_{name}.png"
        low, high = settings["color_limits"][name]
        plt.imsave(path, component(image), cmap=cmap, vmin=low, vmax=high, origin="upper")
        print(path, "pixels:", image.shape[::-1])
